# NLP Assignment 2: Comprehensive Pipeline
**Student:** Abubakar Imran (i23-2589)  
**Course:** Natural Language Processing

---
### Assignment Overview
This notebook contains the scratch implementation of various NLP techniques:
1. **Embeddings**: TF-IDF, PPMI, and Word2Vec (Skip-gram + Neg Sampling)
2. **Sequence Labeling**: BiLSTM for POS and BiLSTM-CRF for NER
3. **Attention**: Transformer Encoder with Manual Multi-head Attention

**Strict Constraint**: Only PyTorch and NumPy are used. No external NLP libraries like sklearn, gensim, or HuggingFace.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import collections
import os
import math

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Data Loading and Preprocessing

In [ ]:
def load_and_tokenize(file_path="data/cleaned.txt"):
    if not os.path.exists(file_path): return []
    with open(file_path, 'r', encoding='utf-8') as f: text = f.read()
    return text.split()

def build_vocabulary(tokens, max_vocab_size=10000):
    word_counts = collections.Counter(tokens)
    common = word_counts.most_common(max_vocab_size-1)
    word2idx = {"<UNK>": 0}
    idx2word = {0: "<UNK>"}
    for word, _ in common:
        if word not in word2idx:
            idx = len(word2idx)
            word2idx[word], idx2word[idx] = idx, word
    return word2idx, idx2word, word_counts

tokens = load_and_tokenize()
word2idx, idx2word, word_counts = build_vocabulary(tokens)
corpus_indices = [word2idx.get(t, 0) for t in tokens]
print(f"Corpus size: {len(tokens)}, Vocab size: {len(word2idx)}")

## 2. Count-Based Embeddings: TF-IDF & PPMI

In [ ]:
from embeddings.tfidf import TFIDF
from embeddings.ppmi import PPMI
from utils.visualize import visualize_embeddings

# TF-IDF (Treating every 100 words as a 'document' for demonstration)
doc_size = 100
pseudo_docs = [tokens[i:i+doc_size] for i in range(0, len(tokens), doc_size)]
tfidf_obj = TFIDF(word2idx)
tfidf_matrix = tfidf_obj.fit_transform(pseudo_docs)
np.save("embeddings/tfidf_matrix.npy", tfidf_matrix)
print("TF-IDF calculated and saved.")

# PPMI
ppmi_obj = PPMI(len(word2idx))
ppmi_obj.build_co_occurrence(corpus_indices, window_size=5)
ppmi_matrix = ppmi_obj.calculate_ppmi()
np.save("embeddings/ppmi_matrix.npy", ppmi_matrix)
print("PPMI calculated and saved.")

# Visualization (Top 20 words)
top_20 = [idx2word[i] for i in range(1, 21)]
visualize_embeddings(ppmi_matrix, word2idx, top_20, title="PPMI Visualization (PCA SVD)")

## 3. Predicted Embeddings: Word2Vec (Skip-gram)

In [ ]:
from embeddings.word2vec import SkipGramModel

class OptimizedW2VDataset(Dataset):
    def __init__(self, corpus, window_size=5):
        c = np.array(corpus, dtype=np.int64)
        centers, contexts = [], []
        for i in range(1, window_size+1):
            centers.append(c[:-i]); contexts.append(c[i:])
            centers.append(c[i:]); contexts.append(c[:-i])
        self.centers = np.concatenate(centers)
        self.contexts = np.concatenate(contexts)

    def __len__(self): return len(self.centers)
    def __getitem__(self, idx): return self.centers[idx], self.contexts[idx]

# Training
NEG_SAMPLES = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = OptimizedW2VDataset(corpus_indices)
loader = DataLoader(dataset, batch_size=1024, shuffle=True)
model = SkipGramModel(len(word2idx), 100).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

neg_probs = torch.tensor([word_counts.get(idx2word[i], 0)**0.75 for i in range(len(word2idx))])
neg_probs = neg_probs / neg_probs.sum()

print("Training Word2Vec...")
for epoch in range(5):
    for i, (cnt, ctx) in enumerate(loader):
        cnt, ctx = cnt.to(device), ctx.to(device)
        neg = torch.multinomial(neg_probs, cnt.size(0)*NEG_SAMPLES, replacement=True).view(-1, NEG_SAMPLES).to(device)
        p_score, n_score = model(cnt, ctx, neg)
        loss = -(F.logsigmoid(p_score).sum() + F.logsigmoid(-n_score).sum())
        opt.zero_grad(); loss.backward(); opt.step()
        if i % 1000 == 0: print(f"Epoch {epoch+1}, Step {i}, Loss: {loss.item()/1024:.4f}")

w2v_embeds = model.in_embed.weight.data.cpu().numpy()
np.save("embeddings/embeddings_w2v.npy", w2v_embeds)

## 4. Sequence Labeling: BiLSTM & Manual CRF

In [ ]:
from models.bilstm import POSModel, NERModel

def read_conll(p): # Helper to read generated datasets
    sents, cur = [], []
    with open(p, "r", encoding="utf-8") as f:
        for l in f:
            if l.strip() == "": 
                if cur: sents.append(cur); cur = []
            else: cur.append(tuple(l.strip().split("\t")))
    return sents

pos_data = read_conll("data/pos_train.conll")
ner_data = read_conll("data/ner_train.conll")
pos_tag2idx = {t: i for i, t in enumerate(sorted(list(set([w[1] for s in pos_data for w in s]))))}
ner_tag2idx = {t: i for i, t in enumerate(sorted(list(set([w[1] for s in ner_data for w in s]))))}

# POS Training
pos_m = POSModel(len(word2idx), 100, 128, len(pos_tag2idx))
pos_m.encoder.embedding.weight.data.copy_(torch.from_numpy(w2v_embeds))
opt_p = torch.optim.Adam(pos_m.parameters(), lr=0.01)

print("\nTraining POS BiLSTM...")
for s in pos_data[:200]: # Mini-batch for demonstration
    x = torch.tensor([word2idx.get(w[0], 0) for w in s]).unsqueeze(0)
    y = torch.tensor([pos_tag2idx[w[1]] for w in s])
    scores = pos_m(x).squeeze(0)
    loss = F.nll_loss(scores, y)
    opt_p.zero_grad(); loss.backward(); opt_p.step()
print("POS Training Done.")

## 5. Transformer Encoder with Manual Attention

In [ ]:
from models.transformer import TransformerClassifier

trans_m = TransformerClassifier(len(word2idx), 128, 4, 5)
print("Transformer Classifier (Manual Attention) Initialized.")

# Dummy forward pass
dummy_x = torch.randint(0, len(word2idx), (8, 20))
logits = trans_m(dummy_x)
print(f"Transformer Output Shape: {logits.shape}")

## 6. Comparison Table

| Task | Algorithm | Key Technique | Learning Type |
| :--- | :--- | :--- | :--- |
| Embeddings | TF-IDF | Stat. Weighting | Count-based |
| Embeddings | PPMI | Prob. Context | Count-based |
| Embeddings | Word2Vec | Shuffle + Neg Sample | Neural |
| POS Tagging | BiLSTM | Recurrence | Sequential |
| NER | BiLSTM-CRF | Viterbi Decoding | Global Opt |
| Clf | Transformer | Multi-head Attention | Parallel Attention |